# IREE Backend 

Starting with migraphx

```
func.func @testing(%first :!migraphx.shaped<10x10xf32, 10x1>, %second: !migraphx.shaped<10x10xf32, 10x1>) -> !migraphx.shaped<10x10xf32, 10x1> {
  %result = migraphx.dot %first, %second: <10x10xf32, 10x1>, <10x10xf32, 10x1> -> <10x10xf32, 10x1>
  func.return %result : !migraphx.shaped<10x10xf32, 10x1>
}
```

We can run to get into linalg using `rocmlir-opt --migraphx-to-linalg <filename>.mlir`:

```
module {
  func.func @testing(%arg0: tensor<100xf32>, %arg1: tensor<100xf32>) -> tensor<100xf32> {
    %cst = arith.constant dense<0.000000e+00> : tensor<1x10x10xf32>
    %expanded = tensor.expand_shape %arg0 [[0, 1, 2]] output_shape [1, 10, 10] : tensor<100xf32> into tensor<1x10x10xf32>
    %expanded_0 = tensor.expand_shape %arg1 [[0, 1, 2]] output_shape [1, 10, 10] : tensor<100xf32> into tensor<1x10x10xf32>
    %0 = linalg.batch_matmul ins(%expanded, %expanded_0 : tensor<1x10x10xf32>, tensor<1x10x10xf32>) outs(%cst : tensor<1x10x10xf32>) -> tensor<1x10x10xf32>
    %collapsed = tensor.collapse_shape %0 [[0, 1, 2]] : tensor<1x10x10xf32> into tensor<100xf32>
    return %collapsed : tensor<100xf32>
  }
}
```

Run the following two compile to run `iree-compile` code on the GPU

```
/opt/iree/bin/iree-compile linalg.mlir --iree-hal-target-device=hip --iree-rocm-target=gfx950 -o result.vmfb  --iree-opt-level=O3
/opt/iree/bin/iree-benchmark-module --module=result.vmfb --device=hip://GPU-62363665-6661-6533-3832-363633636434 --function=testing  --input=100xf32=2 --input=100xf32=3
```

Terminal output:

```
2026-04-07T18:36:31+00:00
Running /opt/iree/bin/iree-benchmark-module
Run on (384 X 1200 MHz CPU s)
CPU Caches:
  L1 Data 48 KiB (x192)
  L1 Instruction 32 KiB (x192)
  L2 Unified 1024 KiB (x192)
  L3 Unified 32768 KiB (x24)
Load Average: 4.09, 4.08, 3.74
***WARNING*** ASLR is enabled, the results may have unreproducible noise in them.
***WARNING*** Library was built as DEBUG. Timings may be affected.
--------------------------------------------------------------------------------------------
Benchmark                                  Time             CPU   Iterations UserCounters...
--------------------------------------------------------------------------------------------
BM_testing/process_time/real_time       1.38 ms         1.85 ms          859 items_per_second=723.4/s
```